# Laboratório 3: DataFrames, Fluxo de Controle e Probabilidade



Bem-vindo ao Laboratório 3! Esta semana, abordaremos mais técnicas de manipulação de DataFrame, condicionais e iteração, e apresentaremos o conceito de aleatoriedade. Você deve concluir todo este laboratório e enviá-lo ao **Moodle** até às **23h59** da data de vencimento.

Consulte as seguintes leituras:
- Agrupamento com subgrupos (ver [BPD 11.4](https://notes.dsc10.com/02-data_sets/groupby.html#subgroups))
- Mesclando DataFrames (veja [BPD 13](https://notes.dsc10.com/02-data_sets/merging.html))
- Declarações condicionais (ver [CIT 9.1](https://inferentialthinking.com/chapters/09/1/Conditional_Statements.html))
- Iteração (ver [CIT 9.2](https://inferentialthinking.com/chapters/09/2/Iteration.html))
- Probabilidade (ver [CIT 9.5](https://inferentialthinking.com/chapters/09/5/Finding_Probabilities.html))

Primeiro, configure os testes e importações executando as células abaixo.

In [1]:
## Descomente para executar no Colab
!pip install babypandas --quiet

In [2]:
import numpy as np
import pandas as pd

# Aqui configuramos a biblioteca de visualização de dados.
import matplotlib
import matplotlib.pyplot as plt
plt.style.use('ggplot')

# 1. Parques Nacionais da Califórnia 🏞️ 🐻

Nesta questão, daremos uma olhada mais de perto nos métodos DataFrame `merge` e `groupby`.

Estaremos trabalhando com dois conjuntos de dados, `california_parks.csv` (armazenado como `parks`) e `california_parks_species.csv` (armazenado como `species`), que fornecem informações sobre os Parques Nacionais da Califórnia e as espécies de plantas e animais encontrados lá , respectivamente. Estes são um subconjunto de um dataset maior, [disponibilizado pela National Parks Services](https://www.kaggle.com/nationalparkservice/park-biodiversity). Também criamos um terceiro DataFrame, `parks_species`, que contém o número de espécies por parque.

Execute a célula abaixo para carregar nossos dados.

In [3]:
parks = pd.read_csv("https://raw.githubusercontent.com/dsc-courses/dsc10-2023-wi/main/labs/lab03/data/california_parks.csv")
species = pd.read_csv("https://raw.githubusercontent.com/dsc-courses/dsc10-2023-wi/main/labs/lab03/data/california_parks_species.csv")
parks_species = pd.DataFrame().assign(
    count=species.groupby('Park Name').count().get('Category')
)

No momento, as informações que temos sobre cada Parque Nacional da Califórnia estão divididas em dois DataFrames. O DataFrame `parks` contém o código, estado, tamanho e localização de cada parque, e o DataFrame `parks_species` contém o número de espécies em cada parque. Execute as células abaixo para ver os dois DataFrames.

In [4]:
parks

,Park Code,Park Name,State,Acres,Latitude,Longitude
0,CHIS,Channel Islands National Park,CA,249561,34.01,-119.42
1,JOTR,Joshua Tree National Park,CA,789745,33.79,-115.90
2,LAVO,Lassen Volcanic National Park,CA,106372,40.49,-121.51
3,PINN,Pinnacles National Park,CA,26606,36.48,-121.16
4,REDW,Redwood National Park,CA,112512,41.30,-124.00
5,SEKI,Sequoia and Kings Canyon National Parks,CA,865952,36.43,-118.68
6,YOSE,Yosemite National Park,CA,761266,37.83,-119.50


In [5]:
parks_species

,count
Park Name,
Channel Islands National Park,1885
Joshua Tree National Park,2294
Lassen Volcanic National Park,1797
Pinnacles National Park,1416
Redwood National Park,6310
Sequoia and Kings Canyon National Parks,1995
Yosemite National Park,2088


**Questão 1.1.** Abaixo, utilize o método `merge` para criar um novo DataFrame chamado `parks_with_species`, que terá as informações existentes dos parques junto com o número de espécies que cada um possui. Certifique-se de que o DataFrame tenha apenas uma linha por parque. Seu DataFrame deve ficar assim:

|    | Park Code   | Park Name                               | State   |   Acres |   Latitude |   Longitude |   count |
|---:|------------|----------------------------------------|--------|--------|-----------|------------|--------|
|  0 | CHIS        | Channel Islands National Park           | CA      |  249561 |      34.01 |     -119.42 |    1885 |
|  1 | JOTR        | Joshua Tree National Park               | CA      |  789745 |      33.79 |     -115.9  |    2294 |
|  2 | LAVO        | Lassen Volcanic National Park           | CA      |  106372 |      40.49 |     -121.51 |    1797 |
|  3 | PINN        | Pinnacles National Park                 | CA      |   26606 |      36.48 |     -121.16 |    1416 |
|  4 | REDW        | Redwood National Park                   | CA      |  112512 |      41.3  |     -124    |    6310 |
|  5 | SEKI        | Sequoia and Kings Canyon National Parks | CA      |  865952 |      36.43 |     -118.68 |    1995 |
|  6 | YOSE        | Yosemite National Park                  | CA      |  761266 |      37.83 |     -119.5  |    2088 |

In [7]:
parks_with_species = parks.merge(parks_species, on='Park Name')
parks_with_species

,Park Code,Park Name,State,Acres,Latitude,Longitude,count
0,CHIS,Channel Islands National Park,CA,249561,34.01,-119.42,1885
1,JOTR,Joshua Tree National Park,CA,789745,33.79,-115.90,2294
2,LAVO,Lassen Volcanic National Park,CA,106372,40.49,-121.51,1797
3,PINN,Pinnacles National Park,CA,26606,36.48,-121.16,1416
4,REDW,Redwood National Park,CA,112512,41.30,-124.00,6310
5,SEKI,Sequoia and Kings Canyon National Parks,CA,865952,36.43,-118.68,1995
6,YOSE,Yosemite National Park,CA,761266,37.83,-119.50,2088


Agora, vamos dar uma olhada no DataFrame `species`. Cada parque tem muitas espécies diferentes, e cada espécie varia em abundância em cada parque.

In [8]:
species

,Park Name,Category,Order,Family,Common Names,Abundance
0,Channel Islands National Park,Mammal,Carnivora,Canidae,Channel Islands Gray Fox,Rare
1,Channel Islands National Park,Mammal,Carnivora,Mephitidae,Spotted Skunk,Uncommon
2,Channel Islands National Park,Mammal,Carnivora,Mustelidae,Sea Otter,NaN
3,Channel Islands National Park,Mammal,Carnivora,Otariidae,Guadalupe Fur Seal,Occasional
4,Channel Islands National Park,Mammal,Carnivora,Otariidae,Northern Fur Seal,Uncommon
...,...,...,...,...,...,...
17780,Yosemite National Park,Vascular Plant,Solanales,Solanaceae,Parish's Nightshade,Rare
17781,Yosemite National Park,Vascular Plant,Solanales,Solanaceae,"Chaparral Nightshade, Purple Nightshade",Uncommon
17782,Yosemite National Park,Vascular Plant,Vitales,Vitaceae,"Thicket Creeper, Virginia Creeper, Woodbine",Rare
17783,Yosemite National Park,Vascular Plant,Vitales,Vitaceae,"California Grape, California Wild Grape",Uncommon


**Questão 1.2.** Usando o método `groupby`, atribua a variável `species_abundance` a um DataFrame que *classifica* os parques por Nome do Parque e Abundância.

_**Dica:**_ Redefina o índice e atribua colunas para que você tenha três colunas: `'Park Name'`, `'Abundance'` e `'Category'`. As primeiras linhas do seu DataFrame devem ficar assim:

|    | Park Name                               | Abundance   |   Category |
|---|----------------------------------------|------------|-----------|
|  0 | Channel Islands National Park           | Abundant    |         48 |
|  1 | Channel Islands National Park           | Common      |        228 |
|  2 | Channel Islands National Park           | Occasional  |        190 |
|  3 | Channel Islands National Park           | Rare        |        368 |
|  4 | Channel Islands National Park           | Uncommon    |        471 |
|  5 | Channel Islands National Park           | Unknown     |        173 |
|  6 | Joshua Tree National Park               | Abundant    |         37 |
|  7 | Joshua Tree National Park               | Common      |        543 |
|  8 | Joshua Tree National Park               | Occasional  |         84 |
|  9 | Joshua Tree National Park               | Rare        |         90 |

In [22]:
# species_abundance = pd.DataFrame().assign(ParkName = species.get('Park Name'), Abundance = species.get('Abundance'), Category = species.get('Category')).groupby(['ParkName', 'Abundance'])['Category'].count().reset_index()
species_abundance = species.groupby(['Park Name', 'Abundance'])['Category'].count().reset_index()
species_abundance

,Park Name,Abundance,Category
0,Channel Islands National Park,Abundant,48
1,Channel Islands National Park,Common,228
2,Channel Islands National Park,Occasional,190
3,Channel Islands National Park,Rare,368
4,Channel Islands National Park,Uncommon,471
5,Channel Islands National Park,Unknown,173
6,Joshua Tree National Park,Abundant,37
7,Joshua Tree National Park,Common,543
8,Joshua Tree National Park,Occasional,84
9,Joshua Tree National Park,Rare,90


## 2. Nachos 🧀 🌶️

Em Python, os valores booleanos podem ser `True` ou `False`. Obtemos valores booleanos ao usar operadores de comparação, entre os quais `<` (menor que), `>` (maior que) e `==` (igual a). Para uma lista mais completa, [veja aqui](https://www.tutorialspoint.com/python/comparison_operators_example.htm).

Execute a célula abaixo para ver um exemplo de operador de comparação em ação.

In [23]:
3 > 1 + 1

True

Podemos até atribuir o resultado de uma operação de comparação a uma variável.

In [ ]:
result = 10 / 2 == 5
result

Matrizes são compatíveis com operadores de comparação. A saída é uma matriz de valores booleanos.

In [ ]:
np.array([1, 5, 7, 8, 3, -1]) > 3

Esperando na mesa de jantar só para você está uma tigela quente de nachos! Digamos que sempre que você comer um nacho, ele terá queijo (cheese), molho (salsa), ambos (both) ou nenhum (neither), que seria apenas uma tortilla simples.

<img src='https://raw.githubusercontent.com/dsc-courses/dsc10-2023-wi/main/labs/lab03/images/nacho.png' width=300>

Usando a chamada de função `np.random.choice(array_name)`, vamos simular a retirada de nachos da tigela aleatoriamente. Comece executando a célula abaixo várias vezes e observe como os resultados mudam.

In [ ]:
nachos = np.array(['cheese', 'salsa', 'both', 'neither'])
np.random.choice(nachos)

Suponha que pegamos dez nachos aleatoriamente e armazenamos os resultados em um array chamado `ten_nachos`.

In [ ]:
ten_nachos = np.array(['neither', 'cheese', 'both', 'both', 'cheese', 'salsa', 'both', 'neither', 'cheese', 'both'])

**Pergunta 2.1.** Encontre a quantidade de nachos apenas com queijo usando código (não *hardcode* sua resposta).

_**Dica:**_ Nossa solução envolve um operador de comparação e a função `np.count_nonzero`.

In [ ]:
number_cheese = ...
number_cheese

**Declarações Condicionais**

Uma instrução condicional é composta de várias linhas de código que permitem ao Python escolher entre diferentes alternativas com base no fato de alguma condição ser verdadeira.

Aqui está um exemplo básico.

```
def sign(x):
    if x > 0:
        return 'Positive'
```

Como a função funciona é que se a entrada `x` for maior que `0`, obtemos a string `'Positive'` de volta.

Se quisermos testar múltiplas condições ao mesmo tempo, usamos o seguinte formato geral.

```
if <if expression>:
    <if body>
elif <elif expression 0>:
    <elif body 0>
elif <elif expression 1>:
    <elif body 1>
...
else:
    <else body>
```

Apenas um dos corpos será executado. Cada expressão `if` e `elif` (else-if) é avaliada e considerada em ordem, começando no topo. Assim que um valor verdadeiro for encontrado (ou seja, uma vez que uma condição for atendida), o corpo correspondente será executado e o restante da expressão será ignorado. Se nenhuma das expressões `if` ou `elif` for verdadeira, então o `else body` será executado. Para obter mais exemplos e explicações, consulte [CIT 9.1](https://inferentialthinking.com/chapters/09/1/Conditional_Statements.html?highlight=else).

**Questão 2.2.** Complete a seguinte instrução condicional para que a string `'More please'` seja atribuída a `say_please` se o número de nachos com queijo em `ten_nachos` for menor que `5`.

In [ ]:
...
    say_please = 'More please'

say_please

**Pergunta 2.3.** Escreva uma função chamada `nacho_reaction` que retorne uma string representando a reação de uma pessoa, com base no tipo de nacho passado. As reações devem ser conforme mostradas na tabela abaixo.

| Type of nacho    | Reaction |
| ----------- | ----------- |
| `cheese`      | `Cheesy!`      |
| `salsa`  | `Spicy!`        |
| `both`      | `Delicious!`      |
| `neither`  | `Boring.`        |

In [ ]:
def nacho_reaction(nacho):
    ...

# Isso é um exemplo de chamada da sua função
spicy_nacho = nacho_reaction('salsa')
spicy_nacho

Agora considere o DataFrame `ten_nachos_reactions` definido abaixo.

In [ ]:
ten_nachos_reactions = pd.DataFrame().assign(Nacho=ten_nachos)
ten_nachos_reactions

**Pergunta 2.4.** Adicione uma coluna chamada `'Reaction'` ao DataFrame `ten_nachos_reactions` que consiste na reação para cada um dos nachos em `ten_nachos`.

_**Dica:**_ Use o método `apply`.

In [ ]:
ten_nachos_reactions = ...
ten_nachos_reactions

**Pergunta 2.5.** Usando o código, encontre o número de reações `'Delicious!'` para os nachos em `ten_nachos_reactions`. Pense em como você poderia encontrar isso usando métodos DataFrame ou usando `np.count_nonzero`.

In [ ]:
num_delicious = ...
num_delicious

**Questão 2.6.** Complete a função `both_or_neither` abaixo. A função recebe como entrada qualquer DataFrame de nachos e reações, com nomes de colunas `'Nacho'` e `'Reaction'`. A função compara o número de nachos com queijo e molho com o número de nachos sem queijo nem molho. Se houver mais nachos com ambos, a função retornará `'These were some yummy nachos!'` e se houver mais nachos com nenhum deles, a função retornará `'These nachos were disappointing.'` Se houver um número igual de cada um. , a função retorna `'These nachos were hit or miss.'`

In [ ]:
def both_or_neither(nacho_df):
    nachos = nacho_df.get('Nacho')
    number_both = ...
    number_neither = ...
    # Agora retorne a string apropriada que descreve a distribuicao dos nachos.
     ...

# Abaixo nós criamos um DataFrame com dados aleatoriamente gerados e testamos sua função nele.
# NÃO troque nada abaixo dessa célula.
# Contudo, você pode criar uma nova célula e avaliar both_or_neither(ten_nachos_reactions)
# para ver se sua função se comporta como você a programou!

np.random.seed(24)
many_nachos = pd.DataFrame().assign(Nacho=np.random.choice(nachos, 250))
many_nachos = many_nachos.assign(Reaction=many_nachos.get('Nacho').apply(nacho_reaction))
result = both_or_neither(many_nachos)
result

## 3. Billy faminto 🍗 🍕🍟
Depois de um longo dia de aula, Billy decide ir jantar no Dirty Birds. O cardápio de hoje traz as quatro comidas favoritas de Billy: asas, pizza, batatas fritas e palitos de mussarela. No entanto, cada prato tem 25% de chance de acabar antes que Billy chegue ao Dirty Birds.

**Observação:** Use Python como sua calculadora. Suas respostas devem ser expressões (como `0,5 ** 2`); não simplifique suas respostas usando uma calculadora externa. Além disso, todas as suas respostas devem ser dadas como decimais entre 0 e 1, não como porcentagens.

**Pergunta 3.1.** Qual é a probabilidade de Billy conseguir comer asas no Dirty Birds?

In [ ]:
wings_prob = ...
wings_prob

**Pergunta 3.2.** Qual é a probabilidade de Billy conseguir comer todos esses quatro alimentos no Dirty Birds?

In [ ]:
all_prob = ...
all_prob

**Pergunta 3.3.** Qual é a probabilidade de Dirty Birds ter ficado sem pelo menos um dos quatro alimentos antes de Billy chegar lá?

In [ ]:
something_is_out = ...
something_is_out

Para compensar seu suprimento imprevisível de comida, Dirty Birds decide realizar um concurso para ganhar alguns brindes gratuitos do HDH Dining. Há uma bolsa com três bolinhas vermelhas, três bolinhas verdes e três bolinhas azuis. Billy tem que tirar três bolinhas de gude **sem reposição**. Para ganhar, todas as três bolinhas sorteadas por Billy devem ser de cores diferentes.

**Pergunta 3.4.** Qual é a probabilidade de Billy vencer a disputa?

_**Dica:**_ Se você estiver preso, comece determinando a probabilidade de que a segunda bola de gude que Billy tira seja diferente da primeira bola de gude que Billy tira.

In [ ]:
winning_prob = ...
winning_prob

## 4. Iteração 🔂
Usando um loop `for`, podemos executar uma tarefa várias vezes. Isso é conhecido como iteração. Aqui, simularemos o desenho de diferentes naipes de um baralho de cartas. 🃏

In [ ]:
suits = np.array(['♣️', '♥️', '♠️', '♦️'])

draws = np.array([])

repetitions = 6

for i in np.arange(repetitions):
    draws = np.append(draws, np.random.choice(suits))

draws

Outro uso da iteração é percorrer um conjunto de valores. Por exemplo, podemos imprimir todas as cores do arco-íris. 🌈

In [ ]:
rainbow = np.array(["red", "orange", "yellow", "green", "blue", "indigo", "violet"])

for color in rainbow:
    print(color)

Podemos ver que a parte recuada do loop `for`, conhecida como corpo, é executada uma vez para cada item em `rainbow`. Observe que o nome `color` é arbitrário; poderíamos substituir ambas as instâncias de `color` na célula acima por qualquer nome de variável válido e o código funcionaria da mesma forma.

Também podemos usar um loop `for` para adicionar uma variável de forma iterativa. Aqui, contamos o número de números pares em uma matriz de números. Cada vez que encontramos um número par em `num_array`, aumentamos `even_count` em 1. Para verificar se um número individual é par, calculamos seu resto quando dividido por 2 usando o operador `%` ([modulus](https://www.freecodecamp.org/news/the-python-modulo-operator-what-does-the-symbol-mean-in-python-solved/#:~:text=The%20%25%20symbol%20in%20Python%20is,basic%20syntax%20is%3A%20a%20%25%20b)).

In [ ]:
num_array = np.array([1, 3, 4, 7, 21, 23, 28, 28, 30])

even_count = 0

for i in num_array:
    if i % 2 == 0:
        even_count = even_count + 1

even_count

**Pergunta 4.1.** Valentina está jogando dardos. 🎯 Seu alvo de dardos contém dez zonas de tamanhos iguais com valores de pontos de 1 a 10. Escreva o código usando `np.random.choice` que simula sua pontuação total após 1000 lançamentos de dardo.

In [ ]:
possible_point_values = ...
tosses = 1000

total_score = ...
for i in range(tosses):
    ...

total_score

**Pergunta 4.2.** Qual é a pontuação média de um dardo lançado por Valentina?

In [ ]:
average_score = ...
average_score

**Pergunta 4.3.** Na célula a seguir, carregamos o texto de _Winnie-the-Pooh_ de A. A. Milne, o livro que vimos na lição de casa 1. Dividimos o texto em palavras individuais e as armazenamos palavras em uma matriz. Usando um loop `for`, atribua `longer_than_four` ao número de palavras no romance que têm mais de 4 letras. Veja [CIT 9.2](https://inferentialthinking.com/chapters/09/2/Iteration.html) se você tiver dúvidas.

_**Dica:**_ Você pode encontrar o número de letras em uma palavra com a função `len`.

In [ ]:
import urllib.request

# Aqui usamos a urllib.request para ler um texto com base em um url. Também usamos a função .decode() para decodifcar o texto no padrão utf-8.

winnie_string = urllib.request.urlopen('https://raw.githubusercontent.com/dsc-courses/dsc10-2023-wi/main/labs/lab03/data/winnie-the-pooh.txt').read().decode('utf-8')
winnie_words = np.array(winnie_string.split())

# ...

# longer_than_four

# Linha de chegada 🏁

Parabéns! Você concluiu o Laboratório 3.

Para enviar sua tarefa:

1. Selecione `Kernel -> Reiniciar e executar tudo` para garantir que você executou todas as células, incluindo as células de teste.
2. Leia o caderno para ter certeza de que está tudo bem e que suas respostas foram computadas.
3. Baixe seu notebook usando `Arquivo -> Baixar como -> Notebook (.ipynb)` e, em seguida, carregue seu notebook para o Moodle.